In [ ]:
# Dataset
# You can download an air-quality dataset from:
# - Kaggle – Air Quality Data :https://www.kaggle.com/datasets/fedesoriano/air-quality-data-set?utm_source=chatgpt.com
# - UCI – Air Quality Dataset :https://archive.ics.uci.edu/dataset/360/air+quality?utm_source=chatgpt.com




# ============================================================
# EXPERIMENT NO. 5
# DATA VISUALIZATION USING MATPLOTLIB
# Analysis of Air Quality Index (AQI) Trends
# Dataset: City_Air_Quality.csv
# ============================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

# STEP 2: Upload the dataset

print("Upload City_Air_Quality.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# STEP 3: Load the dataset

df = pd.read_csv(file_name)

print("\n========== ORIGINAL DATASET ==========")
print(df.head())

# STEP 4: Explore the dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Description:")
print(df.describe(include="all"))

# STEP 5: Clean column names

# Convert column names to lowercase and replace spaces
# and special characters with underscores.

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

# STEP 6: Identify important columns

# Detect common names used for date and AQI columns.

def find_column(possible_names):

    for name in possible_names:
        if name in df.columns:
            return name

    return None


date_col = find_column([
    "date", "datetime", "date_time",
    "timestamp", "day"
])

aqi_col = find_column([
    "aqi", "air_quality_index",
    "airqualityindex"
])

# Detect common pollutant columns.

pollutant_names = {
    "PM2.5": ["pm2_5", "pm25", "pm2_5_concentration"],
    "PM10": ["pm10"],
    "CO": ["co", "carbon_monoxide"],
    "NO2": ["no2", "nitrogen_dioxide"],
    "SO2": ["so2", "sulfur_dioxide"],
    "O3": ["o3", "ozone"]
}

pollutant_columns = {}

for pollutant, possible_names in pollutant_names.items():

    detected = find_column(possible_names)

    if detected is not None:
        pollutant_columns[pollutant] = detected

print("\n========== DETECTED COLUMNS ==========")
print("Date Column:", date_col)
print("AQI Column:", aqi_col)
print("Pollutant Columns:", pollutant_columns)

# STEP 7: Convert date column into datetime format

if date_col is not None:

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    df = df.dropna(subset=[date_col])

    df = df.sort_values(by=date_col)

# STEP 8: Convert AQI and pollutant columns into numeric format

numeric_columns = []

if aqi_col is not None:
    numeric_columns.append(aqi_col)

numeric_columns.extend(pollutant_columns.values())

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# STEP 9: Check and handle missing values

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

for column in numeric_columns:

    df[column] = df[column].fillna(
        df[column].median()
    )

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# STEP 10: Remove duplicate records

print("\nDuplicate Records:", df.duplicated().sum())

df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)

# STEP 11: Display cleaned dataset

print("\n========== CLEANED DATASET ==========")
print(df.head())

# STEP 12: Plot overall AQI trend over time

if date_col is not None and aqi_col is not None:

    plt.figure(figsize=(12, 5))

    plt.plot(
        df[date_col],
        df[aqi_col],
        marker="o",
        label="AQI"
    )

    plt.title("Overall AQI Trend Over Time")
    plt.xlabel("Date")
    plt.ylabel("Air Quality Index")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

else:
    print("Date or AQI column not found. Skipping AQI trend plot.")

# STEP 13: Plot individual pollutant levels over time

if date_col is not None:

    for pollutant, column in pollutant_columns.items():

        plt.figure(figsize=(12, 5))

        plt.plot(
            df[date_col],
            df[column],
            marker="o",
            label=pollutant
        )

        plt.title(pollutant + " Level Over Time")
        plt.xlabel("Date")
        plt.ylabel("Pollutant Level")
        plt.legend()
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# STEP 14: Compare AQI values using a bar chart

if date_col is not None and aqi_col is not None:

    # Calculate average AQI for each date.

    daily_aqi = df.groupby(
        df[date_col].dt.date
    )[aqi_col].mean()

    plt.figure(figsize=(12, 5))

    plt.bar(
        daily_aqi.index.astype(str),
        daily_aqi.values
    )

    plt.title("Average AQI by Date")
    plt.xlabel("Date")
    plt.ylabel("Average AQI")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# STEP 15: Create box plots for pollutant distributions

if len(pollutant_columns) > 0:

    plt.figure(figsize=(10, 6))

    pollutant_data = [
        df[column].dropna()
        for column in pollutant_columns.values()
    ]

    plt.boxplot(
        pollutant_data,
        tick_labels=list(pollutant_columns.keys())
    )

    plt.title("Distribution of Air Pollutants")
    plt.xlabel("Pollutants")
    plt.ylabel("Pollutant Levels")
    plt.grid(True)
    plt.show()

# STEP 16: Scatter plots to explore AQI
# and pollutant relationships.

if aqi_col is not None:

    for pollutant, column in pollutant_columns.items():

        plt.figure(figsize=(7, 5))

        plt.scatter(
            df[column],
            df[aqi_col]
        )

        plt.title(
            "AQI vs " + pollutant
        )

        plt.xlabel(pollutant + " Level")
        plt.ylabel("AQI")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

# STEP 17: Display correlation between AQI and pollutants

if len(numeric_columns) >= 2:

    correlation = df[numeric_columns].corr()

    print("\n========== CORRELATION MATRIX ==========")
    print(correlation)

    # Visualize the correlation matrix using Matplotlib.

    plt.figure(figsize=(9, 6))

    plt.imshow(
        correlation,
        cmap="coolwarm",
        interpolation="nearest",
        aspect="auto"
    )

    plt.colorbar(label="Correlation")

    plt.xticks(
        range(len(correlation.columns)),
        correlation.columns,
        rotation=45
    )

    plt.yticks(
        range(len(correlation.columns)),
        correlation.columns
    )

    plt.title("Correlation Between AQI and Pollutants")

    # Display correlation values inside the heatmap.

    for i in range(len(correlation.columns)):
        for j in range(len(correlation.columns)):

            plt.text(
                j,
                i,
                round(correlation.iloc[i, j], 2),
                ha="center",
                va="center"
            )

    plt.tight_layout()
    plt.show()

# STEP 18: Display basic AQI statistics

if aqi_col is not None:

    print("\n========== AQI STATISTICS ==========")

    print("Average AQI:", round(df[aqi_col].mean(), 2))
    print("Maximum AQI:", df[aqi_col].max())
    print("Minimum AQI:", df[aqi_col].min())

# STEP 19: Save cleaned dataset

df.to_csv(
    "cleaned_city_air_quality.csv",
    index=False
)

print("\n========== EXPERIMENT COMPLETED ==========")
print("Air quality analysis completed successfully.")
print("Cleaned dataset saved as cleaned_city_air_quality.csv")

# STEP 20: Download the cleaned dataset

files.download("cleaned_city_air_quality.csv")
